In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [3]:
# cms_spread_options_single_look = [
#     "QZQHDJR8ZR2C",
# 	"QZSP0NTLKKFJ",
# 	"QZP43JLWTM5W",
# 	"QZVH3T5N6GJ9",
# 	"QZD8FJ9CGMZ2",
# 	"QZH64P6BDDFN",
# 	"QZKC9F9FV3ZV",
# ]
# df[df["UPI Underlier Name"] == "USD-SOFR ICE Swap Rate vs USD-SOFR-COMPOUND"].to_csv("USD-SOFR ICE Swap Rate vs USD-SOFR-COMPOUND sdr trades.csv")

In [4]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions

In [6]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 2, 27, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 2, 27, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

CACHE HIT...: 100%|██████████| 1/1 [00:00<00:00, 30.17it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,2184692020000000201,,NEWT,TRAD,2026-02-27 05:00:14+00:00,False,IR,None,I,True,...,,NaN,,,NaN,None,None,QZ346HS5CZ8C,NA/Swap Flt Flt AUD,AUD-BBSW vs AUD-BBSW
1,2184630165000000201,,NEWT,TRAD,2026-02-27 05:00:17+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZNXC32WWW1N,NA/Swap Fxd Flt AUD,AUD-BBR-BBSW
2,2188054344000001301,2183863986000000201,CORR,,2026-02-27 05:00:19+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZM9Q7C3586N,NA/Swap Flt Flt JPY USD,JPY-TONA-OIS Compound vs USD-SOFR-OIS Compound
3,2184630471000000201,2183878429000000101,REVI,,2026-02-27 05:00:33+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZSP34M4RS41,NA/Swap Fxd Flt AUD,AUD-BBSW
4,2184631133000000101,,NEWT,TRAD,2026-02-27 05:00:52+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZNXC32WWW1N,NA/Swap Fxd Flt AUD,AUD-BBR-BBSW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21610,2200835396000001001,2198870213000000701,EROR,,2026-02-27 23:43:13+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZRGK6GHDWGC,NA/Swap Fxd Flt BRL,BRL-CDI
21611,2200863627000000201,2200649133000000101,MODI,TRAD,2026-02-27 23:44:51+00:00,True,IR,None,N,False,...,,NaN,,,NaN,None,None,QZRGK6GHDWGC,NA/Swap Fxd Flt BRL,BRL-CDI
21612,2200863626000000101,2200840312000000301,MODI,TRAD,2026-02-27 23:44:52+00:00,True,IR,None,N,False,...,,NaN,,,NaN,None,None,QZRGK6GHDWGC,NA/Swap Fxd Flt BRL,BRL-CDI
21613,2200841313000000101,2198861284000000801,EROR,,2026-02-27 23:45:31+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZRGK6GHDWGC,NA/Swap Fxd Flt BRL,BRL-CDI


In [8]:
# df[df["Notional currency-Leg 1"] == "USD"]
capfloor_upifisn = [
    "NA/O Call Epn USD",
    "NA/O P Epn USD",
]

s = df["Option Premium Amount"].astype("string").str.strip()
s = (s
     .str.replace(r"[\$,]", "", regex=True)                 
     .str.replace(r"^\((.*)\)$", r"-\1", regex=True)        
)
df["Option Premium Amount"] = pd.to_numeric(s, errors="coerce")

out = df[
    df["UPI FISN"].isin(capfloor_upifisn)
    & (df["Package indicator"] == False)
    & (df["Action type"] == "NEWT")
    & (df["Option Premium Amount"] > 0)
    & (df["UPI Underlier Name"].str.contains("SOFR"))
]

# out.to_csv("usd_capfloor_sdr_trades_last_week_feb2026.csv")
# ["Unique Product Identifier"].value_counts()

out["Unique Product Identifier"].value_counts()

Unique Product Identifier
QZQJWDQ4V0VJ    28
QZXNP136XML0    22
QZSMPW1BQH94     7
QZTSC5QX8B68     4
QZ76PH5KCDLK     3
QZXQZX92WVW6     2
QZBW56275SNK     1
QZ6736L305DH     1
QZHM38VZ056X     1
QZV7DX3R26B1     1
QZL9GWT1VKHZ     1
QZ57G64S82CQ     1
QZN0GBXGSC30     1
QZTL3H9DBF9V     1
QZP8B8DJGGP3     1
Name: count, dtype: int64

In [5]:
df[
    # (df["UPI Underlier Name"].str.lower().str.contains("vs"))
    # & (df["UPI Underlier Name"].str.lower().str.contains("usd"))
    # & (df["UPI Underlier Name"].str.lower().str.contains("cad"))
    (df["UPI Underlier Name"].str.lower().str.contains("gbp"))
]["UPI Underlier Name"].value_counts()
# .to_csv("02-26-2026-usdcad-xccy-basis-sdr.csv")
# ["UPI Underlier Name"].value_counts()

UPI Underlier Name
GBP-SONIA-COMPOUND                                 1048
GBP-SONIA-OIS Compound                              979
GBP-SONIA-OIS Compound vs USD-SOFR-OIS Compound     160
NA/Swap OIS GBP                                     152
NA/Swap Fxd Flt GBP                                  19
GBP-SONIA-COMPOUND vs USD-SOFR-COMPOUND               7
GBP-WMBA-SONIA-COMPOUND                               5
GBP-SONIA                                             4
GBP-SONIA vs USD-SOFR                                 3
GBP-LIBOR-BBA vs USD-SOFR                             2
GBP-SONIA-OIS Compound vs OTHER                       1
Name: count, dtype: int64

In [6]:
# start = NY_tz.localize(datetime.datetime(2026, 3, 26, 0, 0))
# end = NY_tz.localize(datetime.datetime(2026, 2, 26, 23, 59))

sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, merge_package_legs=False)
# sdf = USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=False, merge_package_legs=False)

PRICING STRADDLES...: 100%|██████████| 2/2 [00:00<00:00, 246.05it/s]


SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 1.5467787414347294e-26, `time`: 0.0028s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 1.5469759566610347e-26, `time`: 0.0039s


PRICING OUTRIGHTS...: 100%|██████████| 47/47 [00:00<00:00, 177.19it/s]


In [8]:
sdf.tail(4)

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,outright_vega01,outright_gamma01,outright_theta1d,matched_ust_maturity,matched_ust_maturity_trade_confidence,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,invoice_swap_ticker
45,NEWT-TRAD,2215165994000000101,2026-03-02 17:08:14+00:00,2026-03-02,2031-02-10,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D Constant 4Y11MxIMM_H2031 ...,15000000.0,USD,False,...,NaN,NaN,NaN,False,<NA>,<NA>,NaN,NaN,2031-03-17,None
46,NEWT-TRAD,2215232564000000201,2026-03-02 17:12:43+00:00,2026-03-02,2030-03-04,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D Constant 4Yx1Y PAYER BERM...,5000000.0,USD,False,...,NaN,NaN,NaN,False,<NA>,<NA>,NaN,NaN,2031-03-10,None
47,NEWT-TRAD,2215263471000000601,2026-03-02 17:13:59+00:00,2026-03-02,2030-03-04,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D Constant 4Yx1Y PAYER BERM...,5000000.0,USD,False,...,NaN,NaN,NaN,False,<NA>,<NA>,NaN,NaN,2031-03-10,None
48,NEWT-TRAD,2215299715000000501,2026-03-02 17:16:27+00:00,2026-03-02,2030-12-04,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D Constant 4Y9Mx3M PAYER BE...,5000000.0,USD,False,...,NaN,NaN,NaN,False,<NA>,<NA>,NaN,NaN,2031-03-11,None
